# Advanced ANN for Churn Modelling

Cross Validation, Manual Grid Search, and Model Checkpoint.

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint


In [ ]:
dataset = pd.read_csv('/content/Churn_Modelling.csv')
X = dataset.iloc[:, 3:13]
y = dataset.iloc[:, 13]
le = LabelEncoder()
X['Gender'] = le.fit_transform(X['Gender'])
X = pd.get_dummies(X, columns=['Geography'], drop_first=True)
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
def create_model(neurons=16, optimizer='adam'):
    model = Sequential()
    model.add(Dense(neurons, activation='relu', input_shape=(X_train.shape[1],)))
    model.add(Dropout(0.2))
    model.add(Dense(neurons, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
neurons_list = [8, 16, 32]
optimizer_list = ['adam', 'rmsprop']
batch_size_list = [32, 64]
best_accuracy = 0
best_params = {}
kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
for neurons in neurons_list:
    for optimizer in optimizer_list:
        for batch_size in batch_size_list:
            scores = []
            for train_idx, val_idx in kfold.split(X_train, y_train):
                model = create_model(neurons, optimizer)
                model.fit(X_train[train_idx], y_train.iloc[train_idx], epochs=20, batch_size=batch_size, verbose=0)
                pred = (model.predict(X_train[val_idx]) > 0.5).astype(int).reshape(-1)
                scores.append(accuracy_score(y_train.iloc[val_idx], pred))
            avg = np.mean(scores)
            if avg > best_accuracy:
                best_accuracy = avg
                best_params = {'neurons': neurons, 'optimizer': optimizer, 'batch_size': batch_size}
print('Best Accuracy:', best_accuracy)
print('Best Parameters:', best_params)


In [ ]:
best_model = create_model(best_params['neurons'], best_params['optimizer'])
checkpoint = ModelCheckpoint('best_ann_model.keras', monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)
best_model.fit(X_train, y_train, validation_split=0.2, epochs=50, batch_size=best_params['batch_size'], callbacks=[checkpoint], verbose=1)
y_pred = (best_model.predict(X_test) > 0.5).astype(int).reshape(-1)
print('Test Accuracy:', accuracy_score(y_test, y_pred))
print('\nConfusion Matrix:\n', confusion_matrix(y_test, y_pred))
print('\nClassification Report:\n', classification_report(y_test, y_pred))
